# Implementing a custom model

There are two interface methods to implement in the child class extending the parent. `_cm_input_def` simply defines a dictionary of inputs that will be assigned to class attributes with the dictionary key being the attribute name in the class. The values of that dictionary will be used for specification of default values and units of the physical quantity, if default is not applicable you can use `np.nan` for the value and deal with the input validation in the `_run` method. The later will be used to compute a cost model output. A full example is provided below showing how to do so.

### Implement a cost model interface

In [36]:
from costmodels.base import CostModel
from costmodels.units import Quant
import numpy as np


class MyCostModel(CostModel):

    @property
    def _cm_input_def(self) -> dict:
        """
        Definition of cost model input. Keys are input names,
        values can be Quant (alias of pint.Quantity), plain
        numbers without a unit assigned or boolean values.
        """
        return {
            "nwt": 0,
            "turbine_cost": Quant(0, "kEUR"),
            "carbon_blades": False,
            "aep": Quant(np.nan, "GWh"),
        }

    def _run(self) -> dict:
        """Method to run the cost model. Compute the cost model output."""
        assert self.nwt > 0
        assert self.turbine_cost > 0
        CAPEX = self.nwt * self.turbine_cost
        if self.carbon_blades:
            CAPEX *= 1.05
        irr = None
        if not np.isnan(self.aep):
            assert self.aep > 0
            eprice = Quant(0.05, "EUR/kWh")  # could be input
            inflation = Quant(10, "%")  # could be input
            opex = Quant(0.02, "EUR")  # could be input; OPEX EUR/year
            lifetime = 20  # could be input
            cashflows = self.cashflows(
                eprice,
                inflation,
                CAPEX,
                opex,
                self.aep,
                lifetime,
            )
            irr = self.irr(cashflows)
        return {
            "CAPEX": CAPEX,
            "IRR": irr,
        }

### Running the model

The below should motivate the static/dynamic variables usage. Number of turbines would more often than not be a static variable and should not be included in the `run` calls.

In [49]:
# static model variables; they can still be changed on the fly but most often one would use them as static
cm = MyCostModel(
    nwt=1000,
    # quantities can be passed with a unit or as plain numbers
    # (in this case the unit is assumed to be the default unit defined in _cm_input_def)
    turbine_cost=Quant(1e6, "EUR"),
)
print(f"Number of wind turbines: {cm.nwt}")
print(f"Cost per wind turbine [explicit units]: {cm.turbine_cost}")

cm = MyCostModel(
    nwt=1000,
    turbine_cost=1e3,  # now it's assumed to be in kEUR
)
print(f"Cost per wind turbine [plain]: {cm.turbine_cost}")

print(f"CAPEX with 1000 turbines: {cm.run()['CAPEX']}")
print("Number of tubines after run 0th call: ", cm.nwt)

# change number of turbines on run call; accomodate dynamic values
print(f"CAPEX with 500 turbines: {cm.run(nwt=500)["CAPEX"]}")

# notice that nwt will be overriden now !!!
print("Number of tubines after run 1st call: ", cm.nwt)

Number of wind turbines: 1000
Cost per wind turbine [explicit units]: 1000.0 kEUR
Cost per wind turbine [plain]: 1000.0 kEUR
CAPEX with 1000 turbines: 1000000.0 kEUR
Number of tubines after run 0th call:  1000
CAPEX with 500 turbines: 500000.0 kEUR
Number of tubines after run 1st call:  500


### Gradients

In [50]:
cm = MyCostModel(nwt=1000, turbine_cost=Quant(1, "MEUR"))

print(f"IRR with 1 TWh production: {cm.run(aep=Quant(1, "TWh"))["IRR"]}")
print(f"IRR with .9 TWh production: {cm.run(aep=Quant(.9, "TWh"))["IRR"]}")

# take gradient of IRR with respect to AEP and turbine cost input variables
cm.grad("IRR", ["aep", "turbine_cost"])

IRR with 1 TWh production: 9.019853828014801 %
IRR with .9 TWh production: 7.97654552237288 %


{'aep': np.float64(0.01079299009424754),
 'turbine_cost': np.float64(-0.009713691084822784)}